# 音乐影响项目：数据准备

运行方式：在 Jupyter 中选择 Python 3，依次运行所有单元格。仅使用 Python 标准库。

本阶段产出数据质量报告、艺术家五年期画像和相邻时期样本；尚不拟合影响模型。原始文件保持不变。输出写入 `output/preparation/`，重复运行会更新这些派生文件。

口径：艺术家 ID 是连接键；身份冲突记录从音乐画像与期际分析中隔离。完全重复的歌曲记录去重。合作作品为每位参与者提供一次作品观测，同时记录分摊权重；因此不同艺术家的观测并非独立。


In [1]:
import ast
import csv
import hashlib
import json
import math
from collections import Counter, defaultdict
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "problems/data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("请在项目目录或其子目录中运行 Notebook")
DATA = ROOT / "problems/data"
OUT = ROOT / "output/preparation"
OUT.mkdir(parents=True, exist_ok=True)
WINDOW = 5
MIN_TRACKS = 3
FEATURES = ["danceability", "energy", "valence", "tempo", "loudness",
            "acousticness", "instrumentalness", "liveness", "speechiness", "duration_ms"]
assert WINDOW > 0 and MIN_TRACKS > 0

def read_csv(name):
    with (DATA / name).open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def save_csv(name, rows, fields):
    with (OUT / name).open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

tables = {name: read_csv(name) for name in
          ["influence_data.csv", "full_music_data.csv", "data_by_artist.csv", "data_by_year.csv"]}
edges = tables["influence_data.csv"]
raw_tracks = tables["full_music_data.csv"]
print({name: len(rows) for name, rows in tables.items()})


{'influence_data.csv': 42770, 'full_music_data.csv': 98340, 'data_by_artist.csv': 5854, 'data_by_year.csv': 100}


## 1. 核对身份、匹配覆盖和汇总口径

相同 ID 对应不同姓名、流派或活跃年代时，先隔离并导出明细，不凭多数票修改身份。`active_start` 是年代，不是影响发生日期；晚开始活跃的影响者关系仅标记，不直接删除。

汇总表数量与逐曲表按参与者展开后的数量分别统计，差异不自动认定为错误。


In [2]:
metadata = defaultdict(set)
parents = defaultdict(set)
for row in edges:
    parents[row["follower_id"]].add(row["influencer_id"])
    for role in ["influencer", "follower"]:
        metadata[row[role + "_id"]].add((
            row[role + "_name"], row[role + "_main_genre"], row[role + "_active_start"]))
ambiguous = {aid for aid, values in metadata.items() if len(values) > 1}
conflicts = [{"artist_id": aid, "artist_name": name, "genre": genre, "active_decade": decade}
             for aid in sorted(ambiguous)
             for name, genre, decade in sorted(metadata[aid])]
save_csv("identity_conflicts.csv", conflicts,
         ["artist_id", "artist_name", "genre", "active_decade"])

def parse_ids(value):
    ids = ast.literal_eval(value)
    if not isinstance(ids, list) or not ids:
        raise ValueError("artists_id 必须是非空列表")
    if any(isinstance(x, bool) or not str(x).isdigit() for x in ids):
        raise ValueError("非法艺术家 ID")
    return sorted(set(map(str, ids)))

raw_credits = Counter()
for row in raw_tracks:
    raw_credits.update(parse_ids(row["artists_id"]))
summary_ids = {r["artist_id"] for r in tables["data_by_artist.csv"]}
comparison = [{"artist_id": r["artist_id"], "artist_name": r["artist_name"],
               "provided_count": int(r["count"]),
               "raw_track_credits": raw_credits[r["artist_id"]]}
              for r in tables["data_by_artist.csv"]]
save_csv("artist_count_comparison.csv", comparison, list(comparison[0]))
audit = {
    "input_rows": {name: len(rows) for name, rows in tables.items()},
    "input_sha256": {name: hashlib.sha256((DATA / name).read_bytes()).hexdigest()
                     for name in tables},
    "columns": {name: list(rows[0]) for name, rows in tables.items()},
    "empty_cells": {name: sum(v.strip() == "" for r in rows for v in r.values())
                    for name, rows in tables.items()},
    "network_ids": len(metadata),
    "ambiguous_ids": sorted(ambiguous),
    "network_summary_overlap": len(set(metadata) & summary_ids),
    "network_ids_missing_summary": sorted(set(metadata) - summary_ids),
    "summary_ids_without_raw_tracks": sorted(summary_ids - set(raw_credits)),
    "summary_count_disagreements": sum(r["provided_count"] != r["raw_track_credits"]
                                       for r in comparison),
    "duplicate_edges": len(edges) - len({(r["influencer_id"], r["follower_id"]) for r in edges}),
    "self_loops": sum(r["influencer_id"] == r["follower_id"] for r in edges),
    "later_start_influencer_edges": sum(int(r["influencer_active_start"]) >
                                      int(r["follower_active_start"]) for r in edges),
}
print(json.dumps({k: v for k, v in audit.items() if k not in
                  ["input_sha256", "columns", "network_ids_missing_summary",
                   "summary_ids_without_raw_tracks"]}, ensure_ascii=False, indent=2))


{
  "input_rows": {
    "influence_data.csv": 42770,
    "full_music_data.csv": 98340,
    "data_by_artist.csv": 5854,
    "data_by_year.csv": 100
  },
  "empty_cells": {
    "influence_data.csv": 0,
    "full_music_data.csv": 0,
    "data_by_artist.csv": 0,
    "data_by_year.csv": 0
  },
  "network_ids": 5603,
  "ambiguous_ids": [
    "303506",
    "893383",
    "957340"
  ],
  "network_summary_overlap": 5602,
  "summary_count_disagreements": 5613,
  "duplicate_edges": 0,
  "self_loops": 0,
  "later_start_influencer_edges": 733
}


## 2. 清洗歌曲并构造艺术家画像

仅去掉整行完全重复记录。非法数值隔离并输出原因；正常的 0、未知调号 -1 不视为缺失。主要画像使用 10 个连续音乐特征，不把调号编码或 popularity 混入距离。

窗口按公历整五年划分，例如 1960–1964；样本首尾可能不完整，输出有效年份与作品数供筛选。画像保留均值和标准差，后续模型需在训练集上估计标准化参数。


In [3]:
seen = set()
tracks = []
rejected = []
duplicate_count = 0
unit_features = set(FEATURES) - {"tempo", "loudness", "duration_ms"}
for index, row in enumerate(raw_tracks, start=2):
    key = tuple(row.items())
    if key in seen:
        duplicate_count += 1
        continue
    seen.add(key)
    try:
        ids = parse_ids(row["artists_id"])
        year = int(row["year"])
        values = {f: float(row[f]) for f in FEATURES}
        if not 1921 <= year <= 2020:
            raise ValueError("年份超出题目范围")
        if not all(math.isfinite(v) for v in values.values()):
            raise ValueError("非有限特征值")
        if any(not 0 <= values[f] <= 1 for f in unit_features):
            raise ValueError("比例特征超出 [0,1]")
        if values["duration_ms"] <= 0 or values["tempo"] < 0:
            raise ValueError("非法时长或速度")
        if int(row["mode"]) not in (0, 1) or not -1 <= int(row["key"]) <= 11:
            raise ValueError("非法调式或调号")
    except (ValueError, TypeError, SyntaxError) as exc:
        rejected.append({"source_csv_row": index, "reason": str(exc)})
        continue
    tracks.append({"source_csv_row": index, "ids": ids, "year": year, **values})

groups = defaultdict(list)
history = defaultdict(list)
for track in tracks:
    for aid in track["ids"]:
        if aid in ambiguous:
            continue
        start = (track["year"] // WINDOW) * WINDOW
        groups[(aid, start)].append(track)
        history[aid].append(track)

panel = []
for (aid, start), songs in sorted(groups.items()):
    meta = next(iter(metadata[aid])) if aid in metadata else ("", "Unknown", "")
    record = {"artist_id": aid, "artist_name": meta[0], "genre": meta[1],
              "period_start": start, "period_end": start + WINDOW - 1,
              "observed_year_min": min(s["year"] for s in songs),
              "observed_year_max": max(s["year"] for s in songs),
              "n_tracks": len(songs),
              "fractional_credits": sum(1 / len(s["ids"]) for s in songs)}
    for f in FEATURES:
        mean = sum(s[f] for s in songs) / len(songs)
        record[f] = mean
        record[f + "_sd"] = (math.sqrt(sum((s[f] - mean)**2 for s in songs) /
                                      (len(songs) - 1)) if len(songs) > 1 else "")
    panel.append(record)
panel_fields = ["artist_id", "artist_name", "genre", "period_start", "period_end",
                "observed_year_min", "observed_year_max", "n_tracks", "fractional_credits"]
panel_fields += [col for f in FEATURES for col in [f, f + "_sd"]]
save_csv("artist_period_features.csv", panel, panel_fields)
save_csv("rejected_tracks.csv", rejected, ["source_csv_row", "reason"])
save_csv("clean_track_features.csv",
         [{**{k: v for k, v in t.items() if k != "ids"},
           "artist_ids": json.dumps(t["ids"])} for t in tracks],
         ["source_csv_row", "artist_ids", "year", *FEATURES])
audit.update(exact_duplicate_tracks=duplicate_count, rejected_tracks=len(rejected),
             clean_track_rows=len(tracks), artist_period_rows=len(panel),
             collaborative_clean_tracks=sum(len(t["ids"]) > 1 for t in tracks))
print({k: audit[k] for k in ["exact_duplicate_tracks", "rejected_tracks",
                            "clean_track_rows", "artist_period_rows"]})


{'exact_duplicate_tracks': 189, 'rejected_tracks': 0, 'clean_track_rows': 98151, 'artist_period_rows': 14491}


## 3. 构造相邻时期样本与影响者历史画像

前后两期均达到最低作品数，并要求至少一个影响者拥有截至当前窗口末的足够历史作品。影响者之间等权，历史作品在各自艺术家内部等权；不会读取下一期作品构造影响者画像。

关系来自事后整理的静态网络，不能据此声称当时关系已形成。当前样本尚未排除双方共享合作作品；正式影响检验必须增加这个敏感性检查。


In [4]:
lookup = {(r["artist_id"], r["period_start"]): r for r in panel}
transitions = []
for current in panel:
    aid, start = current["artist_id"], current["period_start"]
    following = lookup.get((aid, start + WINDOW))
    if (following is None or min(current["n_tracks"], following["n_tracks"]) < MIN_TRACKS
            or aid not in parents):
        continue
    cutoff = start + WINDOW - 1
    profiles = []
    eligible_ids = []
    for parent in sorted(parents[aid] - ambiguous - {aid}):
        earlier = [s for s in history[parent] if s["year"] <= cutoff]
        if len(earlier) >= MIN_TRACKS:
            profiles.append({f: sum(s[f] for s in earlier) / len(earlier) for f in FEATURES})
            eligible_ids.append(parent)
    if not profiles:
        continue
    record = {"artist_id": aid, "genre": current["genre"],
              "period_start": start, "next_period_start": start + WINDOW,
              "history_cutoff_year": cutoff,
              "n_current": current["n_tracks"], "n_next": following["n_tracks"],
              "n_available_influencers": len(profiles),
              "influencer_ids": json.dumps(eligible_ids)}
    for f in FEATURES:
        record[f + "_current"] = current[f]
        record[f + "_next"] = following[f]
        record[f + "_change"] = following[f] - current[f]
        record[f + "_influencer_history"] = sum(p[f] for p in profiles) / len(profiles)
    transitions.append(record)
fields = ["artist_id", "genre", "period_start", "next_period_start", "history_cutoff_year",
          "n_current", "n_next", "n_available_influencers", "influencer_ids"]
fields += [f + suffix for f in FEATURES for suffix in
           ["_current", "_next", "_change", "_influencer_history"]]
save_csv("artist_transitions.csv", transitions, fields)
audit.update(window_years=WINDOW, min_tracks=MIN_TRACKS,
             transition_rows=len(transitions),
             transition_artists=len({r["artist_id"] for r in transitions}),
             transitions_by_genre=dict(Counter(r["genre"] for r in transitions)))
(OUT / "quality_report.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print("期际样本数:", len(transitions))
print("艺术家数:", audit["transition_artists"])
print("流派覆盖:", audit["transitions_by_genre"])
print("输出目录:", OUT)


期际样本数: 2720
艺术家数: 1438
流派覆盖: {'Pop/Rock': 1377, 'Religious': 36, 'Country': 318, 'R&B;': 355, 'Folk': 33, 'Jazz': 160, 'Vocal': 111, 'Comedy/Spoken': 6, 'Electronic': 46, 'Latin': 155, 'Unknown': 1, 'International': 26, 'Reggae': 30, 'Blues': 19, 'Avant-Garde': 2, 'Classical': 10, 'Easy Listening': 8, 'Stage & Screen': 19, 'New Age': 7, "Children's": 1}
输出目录: e:\26_fall\3300prj\归档\output\preparation


## 4. 基本一致性检查与下一步

以下检查确认数量守恒、合作权重、窗口衔接及历史截止时间。下一阶段还需构造排除目标本人的流派环境画像、按时间/艺术家划分验证集，再拟合差分模型。提供的汇总表不会用于填补当前样本缺失，以免混入不同统计范围。


In [ ]:
assert parse_ids("[123, 456, 123]") == ["123", "456"]
assert len(raw_tracks) == duplicate_count + len(rejected) + len(tracks)
assert len(lookup) == len(panel)
assert all(r["artist_id"] not in ambiguous for r in panel)
expected_credits = sum(1 / len(t["ids"]) for t in tracks
                       for aid in t["ids"] if aid not in ambiguous)
assert math.isclose(sum(r["fractional_credits"] for r in panel), expected_credits)
for r in transitions:
    assert r["next_period_start"] - r["period_start"] == WINDOW
    assert r["history_cutoff_year"] < r["next_period_start"]
    assert min(r["n_current"], r["n_next"]) >= MIN_TRACKS
    for f in FEATURES:
        assert math.isclose(r[f + "_change"], r[f + "_next"] - r[f + "_current"])
print("全部一致性检查通过。")
print("汇总表数量差异艺术家数:", audit["summary_count_disagreements"])
print("请先查看 identity_conflicts.csv、artist_count_comparison.csv 和 quality_report.json。")


全部一致性检查通过。
汇总表数量差异艺术家数: 5613
请先查看 identity_conflicts.csv、artist_count_comparison.csv 和 quality_report.json。


: 